In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120 
const aa = 40
const N  = 2_701_767
const I0 = 3
const S0 = 2_701_767 - 3 
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)


Istar_obs = [
2, 6, 11, 14, 17, 23, 31, 38, 43, 46, 74, 91, 119, 138, 193, 255, 257,
324, 372, 412, 422, 407, 411, 450, 408, 394, 371, 416, 425, 388, 387,
369, 386, 365, 328, 314, 335, 298, 323, 300, 280, 285, 273, 254, 253,
211, 209, 232, 203, 217, 199, 206, 217, 182, 173, 176, 154, 166, 157
]
tau = length(Istar_obs)

model_tag_sym = :reciprocal

KMAX_UPPER = 30  


# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [reciprocal_model] Fitting chain 2 (tau=59)
[ Info: [reciprocal] iter 1000/1000000 elapsed=5.5s, rate=0.089, mean=[0.735, 0.00093, 0.342, 0.382], std=[0.0907, 0.000353, 0.0166, 0.0247] [ADAPT]
[ Info: [reciprocal] iter 2000/1000000 elapsed=10.2s, rate=0.082, mean=[0.691, 0.00100, 0.360, 0.372], std=[0.0755, 0.000267, 0.0207, 0.0228] [ADAPT]
[ Info: [reciprocal] iter 3000/1000000 elapsed=14.0s, rate=0.075, mean=[0.683, 0.00102, 0.375, 0.380], std=[0.0636, 0.000227, 0.0260, 0.0217] [ADAPT]
[ Info: [reciprocal] iter 4000/1000000 elapsed=17.9s, rate=0.070, mean=[0.691, 0.00101, 0.384, 0.386], std=[0.0566, 0.000204, 0.0267, 0.0215] [ADAPT]
[ Info: [reciprocal] iter 5000/1000000 elapsed=21.7s, rate=0.071, mean=[0.685, 0.00104, 0.386, 0.373], std=[0.0522, 0.000196, 0.0242, 0.0326] [ADAPT]
[ Info: [reciprocal] iter 6000/1000000 elapsed=25.6s, rate=0.071, mean=[0.680, 0.00107, 0.384, 0.356], std=[0.0491, 0.000191, 0.0235, 0.0455] [ADAPT]
[ Info: [reciprocal] iter 7000/1000000 elapsed=29